In [0]:
from pyspark.sql import functions as F
from datetime import datetime

class BronzeIngestor:
        def __init__(self, indicador_nombre, esquema):
            self.indicador = indicador_nombre
            self.esquema = esquema  # <--- Recibe el esquema por parámetro
            self.catalog = "socioeconomics"
            self.schema_db = "bronze"
            self.tabla_destino = f"{self.catalog}.{self.schema_db}.bronze_{self.indicador}"

        def procesar_archivo(self, ruta_volumen):
            import json
            with open(ruta_volumen, "r", encoding="utf-8") as f:
                datos_raw = json.load(f)
        
            # Usamos el esquema
            df_base = spark.createDataFrame(datos_raw[1], schema=self.esquema)

            final_df = df_base.withColumn("_ingestado_el", F.current_timestamp()) \
                              .withColumn("_archivo_origen", F.lit(ruta_volumen))

            final_df.write.format("delta").mode("overwrite").saveAsTable(self.tabla_destino)
            print(f"✅ Ingesta finalizada: {self.tabla_destino}")